# Gold template: fact table (Databricks)

Builds a fact table from Silver: one row per declared grain, holding only dimension keys and measures.

**Standards this notebook follows**

1. **Declared grain.** `grain` says in words what one row is; `grain_columns` are the dimension keys that
   make a row unique. The run stops if the source has two rows for one grain.
2. **No degenerate or descriptive fields.** Only dimension keys (`<name>_sk`) and numeric measures. Order
   numbers, statuses, names, codes and flags go in a dimension (for example `dim_order`), and the fact keeps
   that dimension's key. The rules cell refuses a measure that is text or is named like an identifier, and
   an existing table with any other column.
3. **Every key resolves.** Each dimension key is looked up by its Silver key; Type 2 dimensions take the
   version valid on the fact's date (`valid_from <= date < valid_to`). A miss gets the Unknown member, `-1`,
   never NULL, and the share of misses per dimension is reported (`max_unknown_ratio` warns).
4. **Rows that can't be placed** (a grain key that is `-1`) go to `<target>_quarantine`, not into the fact.
5. **Late-arriving dimensions.** Each run re-reads `lookback_days` of Silver, so rows that went to `-1`
   find their dimension row once it arrives. For older ones, run with `full_reload`.
6. **Upsert, never overwrite.** `MERGE` on the grain keys, updating only rows whose `_row_hash` changed.
7. **Referential check** after the write: every key exists in its dimension.
8. **Audit columns:** `_row_hash`, `_run_id`, `_loaded_at`. **Layers.** one Unity Catalog catalog per layer: `bronze`, `silver`, `gold`, with keys in `keys`.

## Who supports this notebook

This is an **enterprise template** (`gold-fact` 1.0.0, Databricks). The line between enterprise and domain is drawn by the cells:

| Cells | Marked | Who owns them |
| --- | --- | --- |
| **Parameters** | `# kratosforge:parameters` | The **domain**: table names, keys, columns, measures. Values only, no code. |
| Every other code cell | `# kratosforge:standard` | The **enterprise**: it maintains, fixes and upgrades them. Don't edit them. |
| Markdown cells | none | Anyone: add notes freely. |

While the standard cells are unchanged and no code cell is added, the notebook **conforms** and the enterprise can
take it over: check it with `fabric-desktop templates check <notebook>`, then hand it over with
`fabric-desktop templates handover <notebook> --domain <domain> --owner <you>`.
Editing a standard cell, or adding a code cell, makes it **domain-owned** ad hoc work: it still runs, but the
domain supports it, and the check lists each change. Need logic the template doesn't have? Put it in Silver
(a derived column) or ask the enterprise to add it to the template, so every domain gets it.

In [ ]:
# kratosforge:parameters template=gold-fact platform=databricks version=1.0.0 | The domain fills in this cell: values only, no code.
# Parameters
source_table = "silver.sales.orders"
target_table = "gold.sales.fact_sales"
grain = "One row per sales order"
grain_columns = ["order_sk"]
# One entry per dimension key in the fact:
#   fk       the column in the fact          dim      the dimension table
#   dim_sk   the dimension's key column      dim_key  the dimension column the Silver value matches
#   source   the Silver column to look up    as_of    (Type 2 only) the Silver date that picks the version
dimensions = [
    {"fk": "order_sk", "dim": "gold.sales.dim_order", "dim_sk": "order_sk", "dim_key": "order_sk",
     "source": "order_sk"},
    {"fk": "customer_sk", "dim": "gold.sales.dim_customer", "dim_sk": "customer_sk", "dim_key": "customer_dk",
     "source": "customer_sk", "as_of": "order_date"},
    {"fk": "order_date_sk", "dim": "gold.common.dim_date", "dim_sk": "date_sk", "dim_key": "date",
     "source": "order_date"},
]
measures = {"amount": "decimal(18,2)", "discount": "decimal(18,2)", "net_amount": "decimal(18,2)"}
watermark_column = "_loaded_at"   # Silver's load time
lookback_days = 7                 # re-read this many days of Silver so late dimension rows are picked up
max_unknown_ratio = 0.01          # warn when more than this share of rows point at an Unknown member
run_id = ""          # the pipeline or orchestrator passes its run id; one is made when empty
full_reload = False  # True re-reads the whole source (after a logic change or a backfill)

In [ ]:
# kratosforge:standard template=gold-fact platform=databricks version=1.0.0 cell=setup | Enterprise-managed: change the Parameters cell, not this one.
# Job runs pass parameters to a notebook task as widgets; a widget with a parameter's name overrides it.
for _name in ("run_id", "full_reload", "load_date"):
    if _name in globals():
        dbutils.widgets.text(_name, str(globals()[_name]))
        _value = dbutils.widgets.get(_name)
        globals()[_name] = _value.strip().lower() == "true" if isinstance(globals()[_name], bool) else _value

import re
import time
import uuid
from datetime import datetime, timezone
from functools import reduce

from pyspark.errors import AnalysisException
from pyspark.sql import Window
from pyspark.sql import functions as F

UNKNOWN_SK = -1  # the Unknown member: rows whose key can't be found point here, never at NULL
_TABLE_PART = re.compile(r"[A-Za-z_][A-Za-z0-9_ -]{0,127}")
_COLUMN = re.compile(r"[A-Za-z_][A-Za-z0-9_]{0,127}")
_TYPE = re.compile(r"(?i)(string|boolean|tinyint|smallint|int|bigint|float|double|date|timestamp|decimal\(\d{1,2},\s*\d{1,2}\))")
_RUN_ID = re.compile(r"[A-Za-z0-9_.:-]{1,64}")


def q(name: str) -> str:
    """A table name with each part in backticks. Parameters arrive from pipelines, so they are checked."""
    parts = name.split(".")
    if not 1 <= len(parts) <= 4 or not all(_TABLE_PART.fullmatch(x) for x in parts):
        raise ValueError(f"Not a table name: {name!r}")
    return ".".join(f"`{x}`" for x in parts)


def c(name: str) -> str:
    if not _COLUMN.fullmatch(name):
        raise ValueError(f"Not a column name: {name!r}")
    return f"`{name}`"


def checked_type(dtype: str) -> str:
    if not _TYPE.fullmatch(dtype.strip()):
        raise ValueError(f"Not a supported type: {dtype!r}")
    return dtype.strip()


def table_exists(name: str) -> bool:
    try:
        spark.table(q(name))  # resolved now, so a missing table raises here
        return True
    except AnalysisException:
        return False


def row_hash(columns: list) -> "F.Column":
    """SHA-256 of the columns, NULL-safe: a change in any of them changes the hash."""
    parts = [F.coalesce(F.col(x).cast("string"), F.lit("\u0000")) for x in columns]
    return F.sha2(F.concat_ws("\u001f", *parts), 256)


run_id = run_id or str(uuid.uuid4())
if not _RUN_ID.fullmatch(run_id):
    raise ValueError("run_id may only hold letters, digits and _ . : -")
loaded_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"Run {run_id} at {loaded_at:%Y-%m-%d %H:%M:%S} UTC")

## 1. Fact rules: grain, keys and measures

In [ ]:
# kratosforge:standard template=gold-fact platform=databricks version=1.0.0 cell=step-1 | Enterprise-managed: change the Parameters cell, not this one.
# A fact holds dimension keys and numbers you add up, at one declared grain. Nothing else.
NUMERIC = re.compile(r"(?i)(tinyint|smallint|int|bigint|float|double|decimal\(\d{1,2},\s*\d{1,2}\))")
# Names that say "identifier or description": degenerate dimensions and descriptive fields.
NOT_A_MEASURE = re.compile(r"(?i).*(_id|_no|_num|_number|_code|_key|_bk|_sk|_name|_desc|_description|_status|_type|_flag|_date|_text)$")

problems = []
fks = [d["fk"] for d in dimensions]
if len(set(fks)) != len(fks):
    problems.append("Each dimension key column (fk) can appear only once")
for d in dimensions:
    for field in ("fk", "dim_sk", "dim_key", "source"):
        c(d[field])
    q(d["dim"])
    if not d["fk"].endswith("_sk"):
        problems.append(f"{d['fk']}: name dimension keys <name>_sk, so everyone can tell keys from measures")
for g in grain_columns:
    if g not in fks:
        problems.append(f"Grain column {g} must be a dimension key. Put identifiers in a dimension and use its key.")
if not grain.strip() or not grain_columns:
    problems.append("Declare the grain: a sentence in grain and the keys that make a row unique in grain_columns")
for m, t in measures.items():
    c(m)
    if not NUMERIC.fullmatch(t.strip()):
        problems.append(f"{m} is {t}: a fact holds numbers you add up. Text, dates and flags belong in a dimension.")
    if NOT_A_MEASURE.fullmatch(m):
        problems.append(f"{m} looks like an identifier or a description (a degenerate field). "
                        "Move it to a dimension and keep that dimension's key here.")
if problems:
    raise ValueError("This fact breaks the fact rules:\n- " + "\n- ".join(problems))

audit = ["_row_hash", "_run_id", "_loaded_at"]
expected = [*fks, *measures, *audit]
if table_exists(target_table):
    extra = sorted(set(spark.table(q(target_table)).columns) - set(expected))
    if extra:
        raise ValueError(f"{target_table} has columns the fact rules don't allow: {extra}. Move them to a dimension.")
print(f"Grain: {grain} ({', '.join(grain_columns)})")

## 2. Read Silver and look up every dimension key

In [ ]:
# kratosforge:standard template=gold-fact platform=databricks version=1.0.0 cell=step-2 | Enterprise-managed: change the Parameters cell, not this one.
high_water = None
if table_exists(target_table) and not full_reload:
    high_water = spark.table(q(target_table)).agg(F.max("_loaded_at")).first()[0]

src = spark.table(q(source_table))
if high_water is not None:
    src = src.where(F.col(c(watermark_column)) >= F.lit(high_water) - F.expr(f"INTERVAL {int(lookback_days)} DAYS"))
src.createOrReplaceTempView("fact_source")

joins, keys = [], []
for i, d in enumerate(dimensions):
    a = f"d{i}"
    on = f"{a}.{c(d['dim_key'])} = s.{c(d['source'])} AND {a}.{c(d['dim_sk'])} <> {UNKNOWN_SK}"
    if d.get("as_of"):
        day = f"CAST(s.{c(d['as_of'])} AS DATE)"
        on += f" AND {day} >= {a}.valid_from AND {day} < {a}.valid_to"
    joins.append(f"LEFT JOIN {q(d['dim'])} {a} ON {on}")
    keys.append(f"CAST(COALESCE({a}.{c(d['dim_sk'])}, {UNKNOWN_SK}) AS BIGINT) AS {c(d['fk'])}")

staged = spark.sql(f"""
SELECT {", ".join(keys)},
       {", ".join(f"CAST(s.{c(m)} AS {t}) AS {c(m)}" for m, t in measures.items())}
FROM fact_source s
{chr(10).join(joins)}
""")
staged = (staged.withColumn("_row_hash", row_hash([*fks, *measures]))
                .withColumn("_run_id", F.lit(run_id))
                .withColumn("_loaded_at", F.lit(loaded_at).cast("timestamp"))
                .persist())

## 3. Quarantine rows that can't be placed, check the grain, report Unknowns

In [ ]:
# kratosforge:standard template=gold-fact platform=databricks version=1.0.0 cell=step-3 | Enterprise-managed: change the Parameters cell, not this one.
unplaced = reduce(lambda a, b: a | b, [F.col(c(g)) == UNKNOWN_SK for g in grain_columns])
rejected = staged.where(unplaced)
if rejected.limit(1).count():
    (rejected.withColumn("_reason", F.lit("a grain dimension row is missing"))
        .write.format("delta").mode("append").option("mergeSchema", "true")
        .saveAsTable(q(f"{target_table}_quarantine")))
    print(f"Some rows couldn't be placed at the grain; they are in {target_table}_quarantine")
placed = staged.where(~unplaced)

duplicates = placed.groupBy(*[c(g) for g in grain_columns]).count().where(F.col("count") > 1).take(5)
if duplicates:
    raise RuntimeError(f"Two or more source rows share one grain ({', '.join(grain_columns)}): {duplicates}. "
                       "The grain is wrong, or Silver has duplicates.")

stats = placed.agg(F.count(F.lit(1)).alias("row_count"),
                   *[F.sum(F.when(F.col(c(f)) == UNKNOWN_SK, 1).otherwise(0)).alias(f) for f in fks]).first()
for f in fks:
    ratio = (stats[f] or 0) / stats["row_count"] if stats["row_count"] else 0
    if ratio > max_unknown_ratio:
        print(f"Warning: {ratio:.1%} of rows have {f} = {UNKNOWN_SK} (Unknown). Load that dimension first, "
              "or check the business keys.")
placed.createOrReplaceTempView("fact_staged")

## 4. Upsert into the fact

In [ ]:
# kratosforge:standard template=gold-fact platform=databricks version=1.0.0 cell=step-4 | Enterprise-managed: change the Parameters cell, not this one.
if not table_exists(target_table):
    ddl = ", ".join([*[f"{c(f)} BIGINT NOT NULL" for f in fks],
                     *[f"{c(m)} {checked_type(t)}" for m, t in measures.items()],
                     "`_row_hash` STRING", "`_run_id` STRING", "`_loaded_at` TIMESTAMP"])
    spark.sql(f"CREATE TABLE IF NOT EXISTS {q(target_table)} ({ddl}) USING DELTA CLUSTER BY ({", ".join(c(g) for g in grain_columns)})")

on = " AND ".join(f"t.{c(g)} = s.{c(g)}" for g in grain_columns)
result = spark.sql(f"""
MERGE INTO {q(target_table)} AS t
USING fact_staged AS s
ON {on}
WHEN MATCHED AND t._row_hash <> s._row_hash THEN
    UPDATE SET {", ".join(f"t.{c(n)} = s.{c(n)}" for n in expected if n not in grain_columns)}
WHEN NOT MATCHED THEN
    INSERT ({", ".join(c(n) for n in expected)}) VALUES ({", ".join(f"s.{c(n)}" for n in expected)})
""").first()
print(f"Fact MERGE: {result.asDict() if result else 'done'}")
staged.unpersist()

## 5. Check the grain and every key

In [ ]:
# kratosforge:standard template=gold-fact platform=databricks version=1.0.0 cell=step-5 | Enterprise-managed: change the Parameters cell, not this one.
grain_keys = ", ".join(c(g) for g in grain_columns)
dupes = spark.sql(f"SELECT {grain_keys} FROM {q(target_table)} GROUP BY {grain_keys} HAVING COUNT(*) > 1 LIMIT 5").take(5)
orphans = {}
for d in dimensions:
    missing = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {q(target_table)} f
    LEFT ANTI JOIN {q(d['dim'])} x ON x.{c(d['dim_sk'])} = f.{c(d['fk'])}
    """).first()["n"]
    if missing:
        orphans[d["fk"]] = missing
if dupes or orphans:
    raise RuntimeError(f"{target_table} breaks a fact rule. Duplicate grain: {dupes}. Keys missing from their dimension: {orphans}")
print(f"{target_table}: grain holds and every key is in its dimension")